**[🏠 Course Home](00_START_HERE.ipynb)** | **🤖 [AI-Native](../ai/14_marginal_likelihood_and_model_comparison.ipynb)** | **🐍 [Python](../python/14_marginal_likelihood_and_model_comparison.ipynb)** | **🔵 [R](../r/14_marginal_likelihood_and_model_comparison.ipynb)** | **⬅️ [Prev: Ch 13: Hierarchical Models & Shrinkage](13_hierarchical_models_and_partial_pooling.ipynb)** | **➡️ [Next: Ch 15: Dynamic Memory & Decay (γ)](15_calibrating_bayesian_decay_and_memory.ipynb)**

*🧭 Part III: Applied Modeling, Regression, Hierarchy & Model Comparison — Chapter 14 of 22*

---

# Chapter 14: Likelihood vs. Marginal Likelihood — Occam's Razor, Bayes Factors & Model Comparison
### *The Probability Budget, The Occam Factor, Lindley's Paradox, and Evidence vs. Prediction*

---

## Executive Summary & Navigation Guide

In statistical modeling, we frequently face a fundamental choice:
* *Which of two or more competing scientific hypotheses best explains our data?*
* *How complex should our model be before it begins memorizing noise?*

In frequentist statistics and standard machine learning, model selection often
relies on ad-hoc penalties: subtracting $2k$ (Akaike Information Criterion,
AIC) or $k \\ln n$ (Bayesian Information Criterion, BIC), or evaluating out-of-
sample holdout splits.

In fully Bayesian inference, however, something extraordinary occurs:
**Occam's razor—the principle that simpler explanations are preferred unless
evidence demands complexity—emerges automatically from the laws of
probability.**

This chapter investigates the foundational mathematical distinction between
**Likelihood** $P(\\mathcal{D} \\mid \\theta, \\mathcal{M})$ and **Marginal
Likelihood** (also known as **Bayesian Evidence**) $P(\\mathcal{D} \\mid
\\mathcal{M})$. We will explore why integrating over parameters enforces a
strict "probability budget," derive the Occam factor from the geometry of the
posterior distribution, unpack the counter-intuitive behavior of Lindley's
Paradox, and contrast hypothesis testing via Bayes Factors against predictive
evaluation via out-of-sample cross-validation (LOO-CV).

```
================================================================================
                    THE MODEL COMPARISON LANDSCAPE
================================================================================
       Point Likelihood                 Marginal Likelihood (Evidence)
   P(D | theta_hat, M)                         P(D | M)
--------------------------------------------------------------------------------
 • Evaluates single optimal point       • Integrates over entire prior space
 • Rewards extra parameters (overfits)  • Automatically penalizes complexity
 • "How well can M fit to D?"         • "What is the average performance of M?"
 • Blind to parameter volume            • Governed by probability conservation
================================================================================
```

---

## Part 1: What Is Marginal Likelihood, and How Does It Differ from Likelihood?

To understand Bayesian model comparison, we must distinguish between two
mathematical quantities that sound superficially similar but describe entirely
different concepts.

### 1.1 The Direct Comparison

| Dimension | Point Likelihood $P(\\mathcal{D} \\mid \\theta, \\mathcal{M})$ | Marginal Likelihood $P(\\mathcal{D} \\mid \\mathcal{M})$ |
| :--- | :--- | :--- |
| **Object evaluated** | A single parameter vector $\\theta$ | An entire model class $\\mathcal{M}$ |
| **Mathematical form** | Density at point $\\theta$ | Integral over parameter manifold $\\Theta$ |
| **Parameter dependence** | Explicit function of $\\theta$ | Parameter $\\theta$ is integrated out |
| **Sensitivity to volume** | Blind to parameter space size | Highly sensitive to prior volume |
| **Behavior with complexity** | Always increases with added params | Penalizes superfluous parameter space |
| **Primary purpose** | Parameter estimation (MLE / MAP) | Model selection & hypothesis testing |

### 1.2 The Mathematical Mechanism: Why $\\theta$ Disappears

Consider Bayes' Theorem for parameter estimation within a specific model class
$\\mathcal{M}$:

$$P(\\theta \\mid \\mathcal{D}, \\mathcal{M}) = \\frac{P(\\mathcal{D} \\mid \\theta, \\mathcal{M}) P(\\theta \\mid \\mathcal{M})}{P(\\mathcal{D} \\mid \\mathcal{M})}$$

The denominator is the **Marginal Likelihood**, defined by the definite
integral:

$$P(\\mathcal{D} \\mid \\mathcal{M}) = \\int_{\\Theta} P(\\mathcal{D} \\mid \\theta, \\mathcal{M}) P(\\theta \\mid \\mathcal{M}) \\, d\\theta$$

Notice what happens mathematically:
1. The likelihood $P(\\mathcal{D} \\mid \\theta, \\mathcal{M})$ evaluates how
probable the observed dataset $\\mathcal{D}$ is if the parameter were fixed
at the specific value $\\theta$.
2. The prior $P(\\theta \\mid \\mathcal{M})$ acts as a probability weighting
function, specifying the credibility assigned to $\\theta$ *before* seeing
the data.
3. The integral is a **definite integral** computed across the entire parameter
space $\\Theta$. By the fundamental theorem of calculus, evaluating a
definite integral across the full domain of a variable completely eliminates
that variable.

Consequently, the parameter $\\theta$ vanishes entirely from the result!
What remains is a single scalar: **the probability of having observed data
$\\mathcal{D}$ under the structural assumptions and prior commitments of model
$\\mathcal{M}$ alone.**

### 1.3 Geometric Intuition: Peak Height vs. Total Mass

Geometrically:
* **Maximizing the point likelihood** ($\\max_\\theta P(\\mathcal{D} \\mid
\\theta)$) asks: *"What is the altitude of the absolute highest mountain peak
on this terrain?"* A flexible model with 50 parameters can construct an
infinitesimally thin spire directly beneath the data point, achieving a
towering peak altitude.
* **The marginal likelihood** asks: *"What is the total weighted volume of the
entire mountain range?"* Because the prior weights must integrate to 1, the
model cannot create high altitude everywhere. If it builds a needle-thin
peak surrounded by vast empty valleys, its total integrated volume is
miniscule.

```
       Peak Height (Likelihood) vs. Total Integrated Volume (Evidence)
       
       Likelihood Surface                       Marginal Likelihood
       
   P(D|theta)                               P(D|theta) * P(theta)
       ^         /\                              ^
       |        /  \   <-- MLE Peak              |        /---
       |       /    \                            |       /     /   <-- Weighted
       |      /      \                           |      /       /      Volume
       +-----+--------+----> theta               +-----+---------+----> theta
```

### 1.4 Step-by-Step Worked Numerical Example: The Coin Toss Dilemma

To see this distinction in clear numbers, let us work through a concrete
example by hand.

**Scenario**: We flip a coin $n = 10$ times and observe $k = 7$ heads.
We want to compare two competing scientific hypotheses:
* **Model $\\mathcal{M}_0$ (Fair Coin)**: The coin is physically balanced.
There is no free parameter: $p = 0.5$ exactly.
* **Model $\\mathcal{M}_1$ (Biased Coin)**: The coin may have any physical bias
$p \\in [0, 1]$. In the absence of prior knowledge, we assign a uniform prior
$p \\sim \\text{Beta}(1, 1)$, meaning $P(p \\mid \\mathcal{M}_1) = 1$ for all
$p \\in [0, 1]$.

#### Step A: Evaluate Model $\\mathcal{M}_0$ (Fair Coin)
Since $\\mathcal{M}_0$ has no free parameters, its marginal likelihood is
simply its point likelihood evaluated at $p = 0.5$:

$$P(\\mathcal{D} \\mid \\mathcal{M}_0) = \\binom{10}{7} (0.5)^7 (1 - 0.5)^3 = 120 \\times (0.5)^{10} = \\frac{120}{1024} \\approx 0.117188$$

#### Step B: Evaluate Model $\\mathcal{M}_1$ by Maximum Likelihood
Under $\\mathcal{M}_1$, the maximum likelihood estimate is $\\hat{p} = 7/10 =
0.7$.
The likelihood at this optimal point is:

$$P(\\mathcal{D} \\mid \\hat{p} = 0.7, \\mathcal{M}_1) = \\binom{10}{7} (0.7)^7 (0.3)^3 = 120 \\times 0.0823543 \\times 0.027 \\approx 0.266828$$

If we relied on maximum likelihood (or minimum training loss), we would conclude
that $\\mathcal{M}_1$ is vastly superior:

$$\\frac{P(\\mathcal{D} \\mid \\hat{p}, \\mathcal{M}_1)}{P(\\mathcal{D} \\mid \\mathcal{M}_0)} = \\frac{0.266828}{0.117188} \\approx 2.277$$

The flexible model appears more than twice as likely!

#### Step C: Evaluate Model $\\mathcal{M}_1$ by Marginal Likelihood
Now, let us calculate the true Bayesian evidence by integrating over all
possible values of $p \\in [0, 1]$:

$$P(\\mathcal{D} \\mid \\mathcal{M}_1) = \\int_0^1 \\binom{10}{7} p^7 (1 - p)^3 \\cdot P(p \\mid \\mathcal{M}_1) \\, dp$$

Since $P(p \\mid \\mathcal{M}_1) = 1$:

$$P(\\mathcal{D} \\mid \\mathcal{M}_1) = \\binom{10}{7} \\int_0^1 p^7 (1 - p)^3 \\, dp$$

The integral is Euler's Beta function $\\text{B}(\\alpha, \\beta) =
\\text{B}(7+1, 3+1) = \\text{B}(8, 4)$:

$$\\text{B}(8, 4) = \\frac{\\Gamma(8) \\Gamma(4)}{\\Gamma(12)} = \\frac{7! \\times 3!}{11!} = \\frac{5040 \\times 6}{39916800} = \\frac{30240}{39916800} = \\frac{1}{1320}$$

Now multiply by the binomial coefficient $\\binom{10}{7} = 120$:

$$P(\\mathcal{D} \\mid \\mathcal{M}_1) = 120 \\times \\frac{1}{1320} = \\frac{120}{1320} = \\frac{1}{11} \\approx 0.090909$$

#### Step D: The Revelation
Compare the two marginal likelihoods:
* $P(\\mathcal{D} \\mid \\mathcal{M}_0) = \\frac{120}{1024} \\approx 0.117188$
* $P(\\mathcal{D} \\mid \\mathcal{M}_1) = \\frac{1}{11} \\approx 0.090909$

The **Bayes Factor** comparing the fair coin to the biased coin is:

$$BF_{01} = \\frac{P(\\mathcal{D} \\mid \\mathcal{M}_0)}{P(\\mathcal{D} \\mid \\mathcal{M}_1)} = \\frac{0.117188}{0.090909} = \\frac{1320}{1024} \\approx 1.289$$

Despite the fact that $\\mathcal{M}_1$ achieved a peak likelihood of $0.2668$,
its *average* likelihood across its parameter space was only $0.0909$.
The simple fair-coin hypothesis $\\mathcal{M}_0$ is actually $1.29\\times$ more
probable than the flexible model!

Why? Because $\\mathcal{M}_1$ had to spread its prior belief across regions like
$p = 0.05, 0.10, 0.90, 0.95$ where the data $k=7$ is almost impossible.
Seeing 7 heads out of 10 is mild random variation that easily happens with a
fair coin; it is insufficient evidence to justify introducing a free parameter!

---

## Part 2: The Automatic Bayesian Occam's Razor (The Gambling Budget Analogy)

Why does marginalization inherently penalize complexity without requiring an
artificial penalty term? The answer lies in the **Conservation of Probability**.

### 2.1 The Probability Conservation Law

Every valid probabilistic model $\\mathcal{M}$ generates a probability
distribution over all possible datasets $\\mathcal{D}$ that could ever be
observed. Because probabilities must sum to 1:

$$\\sum_{\\mathcal{D} \\in \\text{All Datasets}} P(\\mathcal{D} \\mid \\mathcal{M}) = 1 \\quad \\text{or} \\quad \\int P(\\mathcal{D} \\mid \\mathcal{M}) \\, d\\mathcal{D} = 1$$

This equality is non-negotiable. Every model, regardless of how simple or
complex, receives **exactly the same total probability budget: 1.0.**

### 2.2 The Gambling Budget Analogy

Imagine two gamblers enter a casino, each holding exactly one \$1.00 chip:
* **The Focused Gambler (Simple Model $\\mathcal{M}_1$)**: Places the entire
\$1.00 chip on a compact cluster of numbers (simple datasets). If the roulette
ball lands in that cluster, the payout is massive! If the ball lands elsewhere,
the payout is 0.
* **The Reckless Gambler (Complex Model $\\mathcal{M}_2$)**: Breaks the \$1.00
chip into 10,000 microscopic fractions of a cent and scatters them across
every single number on the table (can fit any dataset). The reckless gambler
*always* hits a number, but when they do, the payout is microscopic!

```
                    THE PROBABILITY BUDGET CONSERVATION
                    
    Probability Density P(D | M)
          ^
     High |         /\\  <--- Simple Model M1 (Concentrated Budget)
          |        /  \\      Narrow domain, high probability density!
          |       /    \\
          |      /  ||  \\
          |     /   ||   \\
          |    /    ||    \\  _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _ _
     Low  |___/_____||_____\\/   Complex Model M2 (Diffuse Budget) \\___
          +---------||---------------------------------------------> Dataset D
                  Observed D_obs
```

When observed data $\\mathcal{D}_{\\text{obs}}$ falls in the region that the
simple model predicted:
$$P(\\mathcal{D}_{\\text{obs}} \\mid \\mathcal{M}_1) \\gg P(\\mathcal{D}_{\\text{obs}} \\mid \\mathcal{M}_2)$$

The simple model wins decisively not because we artificially punished the
complex model, but because the complex model diluted its predictive budget
over millions of datasets that never occurred!

### 2.3 The Analytical Anatomy of the Occam Factor

We can make this intuition mathematically rigorous using the **Laplace
approximation** to the marginal likelihood integral.

Suppose the posterior distribution $P(\\theta \\mid \\mathcal{D},
\\mathcal{M})$ is peaked around the maximum a posteriori (MAP) estimate
$\\hat{\\theta}$, with Hessian matrix of negative log-posterior curvature:

$$\\mathbf{H} = -\\nabla^2 \\ln P(\\hat{\\theta} \\mid \\mathcal{D}, \\mathcal{M})$$

Expanding the integrand around $\\hat{\\theta}$ yields:

$$P(\\mathcal{D} \\mid \\mathcal{M}) \\approx P(\\mathcal{D} \\mid \\hat{\\theta}, \\mathcal{M}) \\times P(\\hat{\\theta} \\mid \\mathcal{M}) \\times (2\\pi)^{d/2} |\\mathbf{H}|^{-1/2}$$

Let:
* $\\Delta \\theta_{\\text{post}} = (2\\pi)^{d/2} |\\mathbf{H}|^{-1/2}$
  represent
the **posterior uncertainty volume** (the region of parameter space compatible
with the observed data).
* $\\Delta \\theta_{\\text{prior}} = \\frac{1}{P(\\hat{\\theta} \\mid
\\mathcal{M})}$ represent the **prior parameter volume** over which the prior
was spread.

Then the evidence decomposes cleanly into two factors:

$$P(\\mathcal{D} \\mid \\mathcal{M}) \\approx \\underbrace{P(\\mathcal{D} \\mid \\hat{\\theta}, \\mathcal{M})}_{\\text{Best Fit (Likelihood)}} \\times \\underbrace{\\frac{\\Delta \\theta_{\\text{post}}}{\\Delta \\theta_{\\text{prior}}}}_{\\text{Occam Factor } < 1}$$

Taking logarithms:

$$\\ln P(\\mathcal{D} \\mid \\mathcal{M}) \\approx \\ln P(\\mathcal{D} \\mid \\hat{\\theta}, \\mathcal{M}) - \\ln\\left(\\frac{\\Delta \\theta_{\\text{prior}}}{\\Delta \\theta_{\\text{post}}}\\right)$$

The **Occam Factor** $\\frac{\\Delta \\theta_{\\text{post}}}{\\Delta
\\theta_{\\text{prior}}}$ is the fraction of the prior parameter space that
survives after seeing the data. Because learning always shrinks uncertainty,
$\\Delta \\theta_{\\text{post}} < \\Delta \\theta_{\\text{prior}}$, so the Occam
Factor is strictly less than 1.

Every time you add an unconstrained parameter that requires tuning:
1. The prior volume $\\Delta \\theta_{\\text{prior}}$ expands.
2. The ratio $\\frac{\\Delta \\theta_{\\text{post}}}{\\Delta
\\theta_{\\text{prior}}}$ plummets.
3. Unless the best-fit likelihood increases by enough to compensate for this
shrinkage, the marginal likelihood decreases!

---

## Part 3: Bayes Factors ($BF_{12}$), Posterior Model Odds & Lindley's Paradox

### 3.1 Model Odds and the Bayes Factor

When comparing two models $\\mathcal{M}_1$ and $\\mathcal{M}_2$, we apply Bayes'
Theorem at the model level:

$$P(\\mathcal{M}_1 \\mid \\mathcal{D}) = \\frac{P(\\mathcal{D} \\mid \\mathcal{M}_1) P(\\mathcal{M}_1)}{P(\\mathcal{D})}$$

Taking the ratio of posterior probabilities for two models yields:

$$\\underbrace{\\frac{P(\\mathcal{M}_1 \\mid \\mathcal{D})}{P(\\mathcal{M}_2 \\mid \\mathcal{D})}}_{\\text{Posterior Odds}} = \\underbrace{\\frac{P(\\mathcal{M}_1)}{P(\\mathcal{M}_2)}}_{\\text{Prior Odds}} \\times \\underbrace{\\frac{P(\\mathcal{D} \\mid \\mathcal{M}_1)}{P(\\mathcal{D} \\mid \\mathcal{M}_2)}}_{\\text{Bayes Factor } BF_{12}}$$

The **Bayes Factor** $BF_{12}$ is the exact mathematical multiplier that
updates prior model odds into posterior model odds.

### 3.2 Harold Jeffreys' Scale of Evidence

To interpret the numerical magnitude of a Bayes Factor, statistician Harold
Jeffreys proposed a widely adopted calibration scale:

| Bayes Factor $BF_{12}$ | $\\ln(BF_{12})$ | Strength of Evidence for $\\mathcal{M}_1$ |
| :--- | :--- | :--- |
| **$1$ to $3$** | $0$ to $1.1$ | Barely worth mentioning (anecdotal) |
| **$3$ to $10$** | $1.1$ to $2.3$ | Substantial / Moderate evidence |
| **$10$ to $30$** | $2.3$ to $3.4$ | Strong evidence |
| **$30$ to $100$** | $3.4$ to $4.6$ | Very strong evidence |
| **$> 100$** | $> 4.6$ | Decisive evidence |

If $BF_{12} < 1$, taking the reciprocal $1 / BF_{12} = BF_{21}$ provides the
evidence supporting model $\\mathcal{M}_2$.

### 3.3 Lindley's Paradox: The Danger of "Vague" Priors

In parameter estimation (Chapters 1–7), using an extremely wide prior such as
$\\theta \\sim \\mathcal{N}(0, 1000^2)$ is common practice. When sample size
$N$ is moderate, the likelihood dominates and the posterior is virtually
unaffected by the prior width.

**In model comparison, however, this habit is fatal.** This cautionary truth
is known as **Lindley's Paradox** (Lindley, 1957).

#### Mathematical Formulation
Suppose we test a sharp null hypothesis against an open alternative:
* $\\mathcal{M}_0$: $\\theta = 0$ (no effect).
* $\\mathcal{M}_1$: $\\theta \\sim \\mathcal{N}(0, \\sigma_0^2)$ (an effect
  exists,
with prior standard deviation $\\sigma_0$).

We collect $n$ independent observations $y_i \\sim \\mathcal{N}(\\theta,
\\sigma^2)$ where measurement noise $\\sigma$ is known.
The sample mean is $\\bar{y}$, with sampling variance $\\sigma^2 / n$.

The frequentist $z$-test statistic is:
$$z = \\frac{\\bar{y}}{\\sigma / \\sqrt{n}}$$

Suppose $z = 2.58$. The frequentist $p$-value is $p \\approx 0.01$.
A frequentist test decisively rejects the null hypothesis $\\mathcal{M}_0$ at
both the $\\alpha = 0.05$ and $\\alpha = 0.01$ significance levels!

Now calculate the Bayesian evidence under both models:
1. Under $\\mathcal{M}_0$, $\\theta = 0$ is fixed:
$$P(\\bar{y} \\mid \\mathcal{M}_0) = \\mathcal{N}\\left(\\bar{y} \\;\\middle|\\;
0, \\frac{\\sigma^2}{n}\\right) = \\frac{1}{\\sqrt{2\\pi \\sigma^2/n}}
\\exp\\left(-\\frac{z^2}{2}\\right)$$

2. Under $\\mathcal{M}_1$, $\\theta$ is integrated out:
$$\\bar{y} = \\theta + \\epsilon \\implies \\bar{y} \\sim \\mathcal{N}\\left(0,
\\sigma_0^2 + \\frac{\\sigma^2}{n}\\right)$$
$$P(\\bar{y} \\mid \\mathcal{M}_1) = \\frac{1}{\\sqrt{2\\pi (\\sigma_0^2 +
\\sigma^2/n)}} \\exp\\left(-\\frac{\\bar{y}^2}{2(\\sigma_0^2 +
\\sigma^2/n)}\\right)$$

Now compute the Bayes Factor $BF_{01}$ in favor of the null hypothesis:

$$BF_{01} = \\frac{P(\\bar{y} \\mid \\mathcal{M}_0)}{P(\\bar{y} \\mid \\mathcal{M}_1)} = \\sqrt{1 + \\frac{n \\sigma_0^2}{\\sigma^2}} \\exp\\left(-\\frac{z^2}{2} \\cdot \\frac{\\frac{n \\sigma_0^2}{\\sigma^2}}{1 + \\frac{n \\sigma_0^2}{\\sigma^2}}\\right)$$

#### What Happens When $\\sigma_0 \\to \\infty$?
Look at the formula as an analyst attempts to make the prior "completely
uninformative" by letting $\\sigma_0 \\to \\infty$:

$$\\lim_{\\sigma_0 \\to \\infty} BF_{01} = \\lim_{\\sigma_0 \\to \\infty} \\left[ \\sqrt{\\frac{n \\sigma_0^2}{\\sigma^2}} \\exp\\left(-\\frac{z^2}{2}\\right) \\right] = \\infty!$$

As $\\sigma_0$ grows, **the Bayes Factor in favor of the null hypothesis goes to
infinity!**
Even though the data produced $z = 2.58$ ($p = 0.01$, rejecting null), the
Bayes Factor declares that the null hypothesis is infinitely more credible than
the alternative!

#### The Resolution of the Paradox
Why does this happen?
By setting $\\sigma_0 = 1000$, the analyst asserted: *"Prior to this experiment,
I believed $\\theta$ could just as easily be $+800$ or $-900$ as $+2$."*
Having spread the probability budget across the vast range $[-3000, +3000]$, the
probability density that $\\theta$ would land anywhere near the observed effect
size $\\bar{y} \\approx 2.58$ was $1/1000 \\approx 0.001$.

The null model $\\mathcal{M}_0$ placed its entire budget at 0. While 0 is not a
perfect fit for $\\bar{y} = 2.58$, it is orders of magnitude closer than the
vast majority of values contemplated by the hyper-diluted alternative model!

**The Golden Rule of Bayes Factors**:
> *You cannot use improper or arbitrarily diffuse priors for Bayesian model
> comparison with Bayes Factors. The prior width directly and linearly scales
> the evidence penalty.*

---

## Part 4: Bayes Factors vs. Out-of-Sample Cross-Validation (PSIS-LOO & WAIC)

Because Bayes Factors are so sensitive to prior specifications, modern Bayesian
workflow draws a sharp distinction between two different modeling objectives:

```
================================================================================
                    THE TWO MODEL EVALUATION PARADIGMS
================================================================================
  Paradigm 1: Attribution (M-Closed)        Paradigm 2: Generalization (M-Open)
  (The M-Closed Perspective)                 (The M-Open Perspective)
--------------------------------------------------------------------------------
• Question: "Which model is the TRUE       • Question: "Which model PREDICTS
  data-generating mechanism?"                unseen future data best?"
• Primary Metric: Marginal Likelihood &    • Primary Metric: Out-of-Sample ELPD
  Bayes Factors (P(D | M))                   via PSIS-LOO-CV or WAIC
• Assumes: True model is in candidate set  • Assumes: All models are wrong; we
• Sensitive to: Prior domain volume        • Sensitive to: Out-of-sample fit
• Failure mode: Lindley's Paradox with     • Invariant to: Prior dilution on
  diffuse priors                             unconstrained dimensions
================================================================================
```

### 4.1 The Polynomial Test Case

To observe how these paradigms behave under severe overfitting, consider
fitting polynomial regression models of increasing degrees:
* **Degree 1 (Linear)**: Underfits the true physical curvature.
* **Degree 2 (Quadratic)**: Matches the true physical generating process.
* **Degree 3 (Cubic)**: Mildly overparameterized; fits noise slightly.
* **Degree 7 (Septic)**: Wildly overparameterized; oscillates violently.

### 4.2 How the Three Metrics Respond

When we evaluate these four candidate models across three metrics:

1. **In-Sample Training Log-Likelihood $\\ln P(\\mathcal{D} \\mid
   \\hat{\\theta})$**:
Monotonically increases with polynomial degree. Degree 7 achieves the highest
training likelihood because extra degrees of freedom can always bend to
minimize training residuals. This is the classic **overfitting trap**.
2. **Log Marginal Likelihood $\\ln P(\\mathcal{D} \\mid \\mathcal{M})$**:
Peaks sharply at **Degree 2**! Degrees 3 and 7 suffer large Occam penalties
because their higher-order coefficients $\\beta_3, \\dots, \\beta_7$ have
prior volume that is wasted on non-existent curvature.
3. **Leave-One-Out Cross-Validation (LOO-CV)**:
Also peaks at **Degree 2**! When a single point is held out, the wild
oscillations of Degree 7 produce catastrophic out-of-sample prediction
errors, dragging its expected log pointwise predictive density (ELPD) down.

Both Marginal Likelihood and Cross-Validation successfully identify Degree 2 as
optimal, but they do so via entirely different mathematical principles:
* Marginal Likelihood penalizes Degree 7 by integrating over the prior parameter
volume.
* Cross-Validation penalizes Degree 7 by testing its empirical forecasts on
held-out observations.

---

## Part 5: Summary & Practical Decision Framework

When should an engineer or data scientist use Bayes Factors, and when should
they use Leave-One-Out Cross-Validation?

```
                      PRACTICAL DECISION FLOWCHART
                      
                      What is your primary goal?
                                  |
            +---------------------+---------------------+
            |                                           |
    Hypothesis Testing                           Predictive Utility
    "Did treatment have effect?"             "Forecast tomorrow's metrics"
            |                                           |
    Do you have informative,                            |
    substantively elicited priors?                      |
            |                                           |
      +-----+-----+                                     |
     Yes          No                                    |
      |            |                                    |
  Compute      Avoid Bayes Factors;                     |
  Bayes        Prior width dominates.                   |
  Factor       Use Cross-Validation                     |
  (BF_12)      or estimation instead.                   |
      |            |                                    |
      +------------+------------------------------------+
                                  |
                                  v
                        Compute PSIS-LOO / WAIC
                    (Expected Predictive Accuracy)
```

### Key Takeaways for Production Practitioners:
1. **Never judge a model by training likelihood alone.** Maximum likelihood
is blind to complexity and guarantees overfitting.
2. **Marginal likelihood averages, it does not maximize.** By integrating over
all parameter values weighted by the prior, models that allocate probability
to barren regions are naturally punished.
3. **The Occam Factor is the ratio of posterior to prior volume.** Simpler
models predict fewer datasets with higher precision; complex models predict
everything with zero confidence.
4. **Vague priors ruin Bayes Factors.** Never use $\\mathcal{N}(0, 1000^2)$
with Bayes Factors unless you want the null model to win automatically
(Lindley's Paradox).
5. **Use LOO-CV when predicting future data.** Leave-One-Out Cross-Validation
(and PSIS-LOO) evaluates models on their actual predictive performance,
making it the gold standard when predicting new outcomes.

---
## 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](00_START_HERE.ipynb)**
* ⬅️ **Previous**: **[Ch 13: Hierarchical Models & Shrinkage](13_hierarchical_models_and_partial_pooling.ipynb)** *(Part III: Applied Modeling, Regression, Hierarchy & Model Comparison)*
* ➡️ **Next**: **[Ch 15: Dynamic Memory & Decay (γ)](15_calibrating_bayesian_decay_and_memory.ipynb)** *(Part IV: Dynamic Systems, Decision Theory & Predictive Guarantees)*
* 🔄 **Companion Tracks for Chapter 14**: [📖 Conceptual](../conceptual/14_marginal_likelihood_and_model_comparison.ipynb) | [🤖 AI-Native](../ai/14_marginal_likelihood_and_model_comparison.ipynb) | [🐍 Python](../python/14_marginal_likelihood_and_model_comparison.ipynb) | [🔵 R](../r/14_marginal_likelihood_and_model_comparison.ipynb)
